# Llama 3.2 3B Fine-tuning with Ultrachat Dataset

This notebook fine-tunes Llama 3.2 3B model using the Ultrachat dataset with Unsloth for efficient training.

## Setup

Install Unsloth library and dependencies.

In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [2]:
import torch
print(f"Current PyTorch version: {torch.__version__}")

Notebook正在使用的版本: 2.9.0+cu128


## Model Configuration

Load Llama 3.2 3B with 4-bit quantization and configure LoRA parameters.

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = torch.bfloat16 # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
 "unsloth/Meta-Llama-3.1-8B-bnb-4bit", # Llama-3.1 2x faster
 "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
 "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
 "unsloth/Meta-Llama-3.1-405B-bnb-4bit", # 4bit for 405b!
 "unsloth/Mistral-Small-Instruct-2409", # Mistral 22b 2x faster!
 "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
 "unsloth/Phi-3.5-mini-instruct", # Phi-3.5 2x faster!
 "unsloth/Phi-3-medium-4k-instruct",
 "unsloth/gemma-2-9b-bnb-4bit",
 "unsloth/gemma-2-27b-bnb-4bit", # Gemma 2x faster!

 "unsloth/Llama-3.2-1B-bnb-4bit", # NEW! Llama 3.2 models
 "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
 "unsloth/Llama-3.2-3B-bnb-4bit",
 "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

 "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
 model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
 max_seq_length = max_seq_length,
 dtype = dtype,
 load_in_4bit = load_in_4bit,
 # token="YOUR_HF_TOKEN_HERE", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/jovyan/miniconda3/envs/unsloth_lab2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jovyan/.local/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA H100 80GB HBM3 MIG 1g.20gb. Num GPUs = 1. Max memory: 19.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Dataset Loading

Load and process the Ultrachat dataset for training.

In [4]:
model = FastLanguageModel.get_peft_model(
 model,
 r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
 target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
 "gate_proj", "up_proj", "down_proj",],
 lora_alpha = 16,
 lora_dropout = 0, # Supports any, but = 0 is optimized
 bias = "none", # Supports any, but = "none" is optimized
 # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
 use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
 random_state = 3407,
 use_rslora = False, # We support rank stabilized LoRA
 loftq_config = None, # And LoftQ
)

Unsloth 2025.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## Training Setup

Configure training parameters including learning rate, batch size, and optimization settings.

In [6]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, DatasetDict # DatasetDict
import torch
import random
# Model loaded ( 3) tokenizer

print("--- ---")
# 'conversations' 
def formatting_prompts_func(examples):
 convos = examples["conversations"] # 
 texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
 return { "text" : texts, }
pass

# --- ---
NEW_DATASET_ID = "aeolian83/HuggingFaceH4-ultrachat_200k_filtered"
TARGET_SIZE = 100000 
SPLIT_NAME = "train_sft" # split 

print(f"\n {NEW_DATASET_ID}")

# 1. 
full_dataset = load_dataset(NEW_DATASET_ID, split = SPLIT_NAME) 

# 2. 
current_size = len(full_dataset)
print(f": {current_size} : {TARGET_SIZE} ")

if current_size > TARGET_SIZE:
 # dataset 
 print("...")
 dataset_dict = full_dataset.train_test_split(
 test_size=TARGET_SIZE, 
 seed=42 # 
 )
 dataset = dataset_dict["test"] # 100k dataset
 
elif current_size < TARGET_SIZE:
 print(" ")
 dataset = full_dataset
else:
 # 
 dataset = full_dataset
 
# 3. ( 'conversations')
if 'messages' in dataset.column_names:
 dataset = dataset.rename_column("messages", "conversations")
 print(" 'conversations'")
else:
 print(" 'conversations' 'messages' ")

# 4. dataset Dataset 
if isinstance(dataset, DatasetDict):
 dataset = dataset["train"] # DatasetDict train split

print(f": {len(dataset)} ")

--- 定义格式化函数 ---

🚀 正在加载新数据集：aeolian83/HuggingFaceH4-ultrachat_200k_filtered
原始数据集大小: 118977 条。目标大小: 100000 条。
正在进行随机抽样...
✅ 对话列已重命名为 'conversations'。
最终训练集大小: 100000 条。


## Training Execution

Run the fine-tuning process.

In [7]:
print(full_dataset.column_names)
# dataset 
# print(dataset.column_names)

['prompt', 'prompt_id', 'messages']


## Model Inference

Test the fine-tuned model with sample prompts.

In [8]:
# 2: map 
if 'messages' in dataset.column_names:
 dataset = dataset.rename_column("messages", "conversations")
 print(" 'conversations'")

from unsloth.chat_templates import standardize_sharegpt

# 1. ShareGPT human/gpt user/assistant
# standardize_sharegpt 
dataset = standardize_sharegpt(dataset) 

# 2. formatting_prompts_func 
# formatting_prompts_func 'messages'
# standardize_sharegpt 

# map
# standardize_sharegpt 
# dataset ['prompt', 'prompt_id', 'messages'] map
dataset = dataset.map(formatting_prompts_func, batched = True,)

Setting TOKENIZERS_PARALLELISM=false for forked processes.
[datasets.arrow_dataset|WARNING]Setting TOKENIZERS_PARALLELISM=false for forked processes.
Map: 100%|██████████| 100000/100000 [00:08<00:00, 11649.55 examples/s]


## Export

Save model in multiple formats (HuggingFace, GGUF, quantized variants).

In [9]:
dataset[5]["conversations"]

[{'content': 'How does Chances incorporate local vendors and businesses into the event experience?',
  'role': 'user'},
 {'content': 'I do not have access to specific information on how chances incorporate local vendors and businesses into the event experience. However, here are some possible ways in which an event organizer can incorporate local vendors and businesses into the event experience: \n\n1. Partnering with local businesses to provide food and beverage options during the event. This can include food trucks or themed food stations run by local restaurants or caterers. \n\n2. Showcasing local craftsmanship or products in a vendor market or artisan alley. This allows attendees to support local businesses and purchase unique souvenirs or gifts. \n\n3. Collaborating with local service providers, such as photographers or event planners, to showcase their work and services to potential clients. \n\n4. Incorporating local entertainment options, such as musicians or performers, into 

In [10]:
dataset[5]["text"]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow does Chances incorporate local vendors and businesses into the event experience?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nI do not have access to specific information on how chances incorporate local vendors and businesses into the event experience. However, here are some possible ways in which an event organizer can incorporate local vendors and businesses into the event experience: \n\n1. Partnering with local businesses to provide food and beverage options during the event. This can include food trucks or themed food stations run by local restaurants or caterers. \n\n2. Showcasing local craftsmanship or products in a vendor market or artisan alley. This allows attendees to support local businesses and purchase unique souvenirs or gifts. \n\n3. Collaborating with local service prov

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
 model = model,
 tokenizer = tokenizer,
 train_dataset = dataset,
 dataset_text_field = "text",
 max_seq_length = max_seq_length,
 data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
 dataset_num_proc = 2,
 packing = False, # Can make training 5x faster for short sequences.
 args = TrainingArguments(
 per_device_train_batch_size = 32,
 gradient_accumulation_steps = 1,
 warmup_steps = 10,
 num_train_epochs = 1, # Set this for 1 full training run.
 # max_steps = None,
 learning_rate = 2e-4,
 fp16 = not is_bfloat16_supported(),
 bf16 = is_bfloat16_supported(),
 logging_steps = 1,
 optim = "adamw_8bit",
 weight_decay = 0.01,
 lr_scheduler_type = "linear",
 seed = 3407,
 output_dir = "outputs",
 report_to = "none", # Use this for WandB etc
 ),
)

Setting TOKENIZERS_PARALLELISM=false for forked processes.
[datasets.arrow_dataset|WARNING]Setting TOKENIZERS_PARALLELISM=false for forked processes.
Unsloth: Tokenizing ["text"] (num_proc=228): 100%|██████████| 100000/100000 [00:41<00:00, 2414.49 examples/s]


In [12]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
 trainer,
 instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
 response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Setting TOKENIZERS_PARALLELISM=false for forked processes.
[datasets.arrow_dataset|WARNING]Setting TOKENIZERS_PARALLELISM=false for forked processes.
Map (num_proc=64): 100%|██████████| 100000/100000 [00:04<00:00, 24946.58 examples/s]


In [13]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow does Chances incorporate local vendors and businesses into the event experience?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nI do not have access to specific information on how chances incorporate local vendors and businesses into the event experience. However, here are some possible ways in which an event organizer can incorporate local vendors and businesses into the event experience: \n\n1. Partnering with local businesses to provide food and beverage options during the event. This can include food trucks or themed food stations run by local restaurants or caterers. \n\n2. Showcasing local craftsmanship or products in a vendor market or artisan alley. This allows attendees to support local businesses and purchase unique souvenirs or gifts. \n\n3. Collaborating with l

In [14]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                  I do not have access to specific information on how chances incorporate local vendors and businesses into the event experience. However, here are some possible ways in which an event organizer can incorporate local vendors and businesses into the event experience: \n\n1. Partnering with local businesses to provide food and beverage options during the event. This can include food trucks or themed food stations run by local restaurants or caterers. \n\n2. Showcasing local craftsmanship or products in a vendor market or artisan alley. This allows attendees to support local businesses and purchase unique souvenirs or gifts. \n\n3. Collaborating with local service providers, such as photographers or event planners, to showcase their work and services to potential clients. \n\n4. Incorporating local entertainment options, such as musicians or performers, into the event program. \n\n5. Working with local transportation companies to provide sh

In [15]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA H100 80GB HBM3 MIG 1g.20gb. Max memory = 19.625 GB.
3.07 GB of memory reserved.


In [16]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 3,125
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.203500
2,1.209700
3,1.368000
4,1.275500
5,1.276800
6,1.244900
7,1.242600
8,1.085100
9,1.185900
10,1.263700


In [17]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

24539.7063 seconds used for training.
409.0 minutes used for training.
Peak reserved memory = 7.697 GB.
Peak reserved memory for training = 4.627 GB.
Peak reserved memory % of max memory = 39.22 %.
Peak reserved memory for training % of max memory = 23.577 %.


In [18]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
 tokenizer,
 chat_template = "llama-3.2",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
 {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
 messages,
 tokenize = True,
 add_generation_prompt = True, # Must add for generation
 return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
 temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContinue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n13, 21, 34, 55, 89, 144, 233, 377, 610, 985, 1597, 2584, 4181.<|eot_id|>']

In [19]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
 {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
 messages,
 tokenize = True,
 add_generation_prompt = True, # Must add for generation
 return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
 use_cache = True, temperature = 1.5, min_p = 0.1)

11, 18, 29, 47, 76, 123, 199, 317, 512, 821, 1319, 2129, 3395, 5461, 8772.<|eot_id|>


In [20]:
import time
import torch

# 1. 
test_messages = [
 {"role": "user", "content": "Explain the difference between supervised, unsupervised, and reinforcement learning in simple terms, focusing on how each type learns."},
]

# 2. 
FastLanguageModel.for_inference(model)

# CUDA Notebook GPU
test_inputs = tokenizer.apply_chat_template(
 test_messages,
 tokenize = True,
 add_generation_prompt = True,
 return_tensors = "pt",
).to("cuda") 

# 3. 
N_RUNS = 5 
total_time = 0.0

print(f"--- Running Inference Speed Test ({N_RUNS} runs) ---")

for i in range(N_RUNS):
 start_time = time.time()
 
 # token 
 # TextStreamer generate
 _ = model.generate(input_ids = test_inputs, max_new_tokens = 256, use_cache = True, 
 temperature = 0.7, top_p = 0.9, do_sample = True)
 
 end_time = time.time()
 run_time = end_time - start_time
 total_time += run_time
 print(f"Run {i+1} time: {run_time:.2f} seconds")

# 4. 
average_time = total_time / N_RUNS
print(f"=====================================================")
print(f"Model: {model.config._name_or_path} ({model.config.num_hidden_layers} layers)")
print(f"Average Inference Time over {N_RUNS} runs: {average_time:.2f} seconds")
print(f"=====================================================")

--- Running Inference Speed Test (5 runs) ---
Run 1 time: 4.22 seconds
Run 2 time: 3.42 seconds
Run 3 time: 4.00 seconds
Run 4 time: 5.52 seconds
Run 5 time: 2.46 seconds
Model: unsloth/llama-3.2-3b-instruct-bnb-4bit (28 layers)
Average Inference Time over 5 runs: 3.92 seconds


In [21]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token="YOUR_HF_TOKEN_HERE") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token="YOUR_HF_TOKEN_HERE") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

In [22]:
if False:
 from unsloth import FastLanguageModel
 model, tokenizer = FastLanguageModel.from_pretrained(
 model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
 max_seq_length = max_seq_length,
 dtype = dtype,
 load_in_4bit = load_in_4bit,
 )
 FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
 {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
 messages,
 tokenize = True,
 add_generation_prompt = True, # Must add for generation
 return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
 use_cache = True, temperature = 1.5, min_p = 0.1)

I am incapable of having physical or visual experiences, but based on the knowledge and data, the tall tower in the capital of france is the Eiffel tower. The eiffel tower is approximately 324 meters (1,063 feet) high and was designed as an entrance arch for the 1889 world's fair in paris. The tower is composed of wrought iron latticework, with a network of beams, columns, and stairs supporting it. At night, the eiffel tower is illuminated by thousands of LED lights and is a iconic symbol of parism.<|eot_id|>


In [23]:
# ==========================================
# Llama-3.2 GGUF (All-in-One )
# Training completed GGUF
# ==========================================

import os
import sys
from huggingface_hub import HfApi

# Hugging Face 
HF_USERNAME = "Marcus719" 
REPO_NAME = "Llama-3.2-3B-changedata-Lab2-GGUF" 
token="YOUR_HF_TOKEN_HERE"YOUR_HF_TOKEN_HERE"" # Write Token
# 

# -----------------------------------------------------------
# Step 0: 
# -----------------------------------------------------------
print(" Step 0: (gguf, protobuf, cmake)...")
# --user Permission denied 
!{sys.executable} -m pip install --user --upgrade gguf protobuf sentencepiece mistral_common numpy cmake huggingface_hub transformers

# cmake
user_bin = os.path.expanduser("~/.local/bin")
if user_bin not in os.environ['PATH']:
 os.environ['PATH'] += f":{user_bin}"

# -----------------------------------------------------------
# Step 1: ( Tokenizer )
# -----------------------------------------------------------
print("\n Step 1: ./model_local ...")
# tokenizer.model/json config.json
model.save_pretrained_merged("model_local", tokenizer, save_method="merged_16bit")
print(" ")

# -----------------------------------------------------------
# Step 2: llama.cpp ( libcurl )
# -----------------------------------------------------------
print("\n Step 2: llama.cpp ( CURL )...")
if os.path.exists("llama.cpp"):
 !rm -rf llama.cpp # 

!git clone https://github.com/ggerganov/llama.cpp
# -DLLAMA_CURL=OFF sudo apt install 
!cd llama.cpp && mkdir build && cd build && cmake .. -DLLAMA_CURL=OFF && cmake --build . --config Release -j --target llama-quantize llama-cli
print(" llama.cpp ")

# -----------------------------------------------------------
# Step 3: FP16 GGUF
# -----------------------------------------------------------
print("\n Step 3: HF GGUF (FP16)...")
# model_local 
!{sys.executable} llama.cpp/convert_hf_to_gguf.py ./model_local \
 --outtype f16 \
 --outfile model_f16.gguf
print(" ")

# -----------------------------------------------------------
# Step 4: 4-bit (Q4_K_M)
# -----------------------------------------------------------
print("\n Step 4: 4-bit ( CPU )...")
# 
quantize_bin = "llama.cpp/build/bin/llama-quantize"
if not os.path.exists(quantize_bin):
 quantize_bin = "llama.cpp/llama-quantize" # 

final_filename = "unsloth.Q4_K_M.gguf"
!{quantize_bin} model_f16.gguf {final_filename} Q4_K_M

if os.path.exists(final_filename):
 size_gb = os.path.getsize(final_filename) / (1024**3)
 print(f" : {size_gb:.2f} GB")
else:
 raise RuntimeError(" ")

# -----------------------------------------------------------
# Step 5: Hugging Face
# -----------------------------------------------------------
print("\n Step 5: Hugging Face...")
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

try:
 api = HfApi(token=HF_TOKEN)
 # 
 api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")
 
 # 
 print(f" : {repo_id}")
 print(f" : {final_filename} ()...")
 api.upload_file(
 path_or_fileobj=final_filename,
 path_in_repo=final_filename,
 repo_id=repo_id,
 repo_type="model"
 )
 print("\n ")
 print(f" : https://huggingface.co/{repo_id}")
 print(" Hugging Face Space (Gradio) ")
 
except Exception as e:
 print(f"\n : {e}")
 print(": Token Write ")

⚙️ Step 0: 正在安装转换工具依赖 (gguf, protobuf, cmake)...
  Using cached huggingface_hub-1.2.1-py3-none-any.whl.metadata (13 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached typer_slim-0.20.0-py3-none-any.whl.metadata (16 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.5 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.57.2,>=4.51.3, but you have transformers 4.57.3 which is incompatible.
unsloth 2025.11.4 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,<=4.57.2,>=4.51.3, but you have transformers 4.57.3 which is incompatible.

💾 Step 1: 正在将模型合并并保存到本地 ./model_local 目录...
Found HuggingFace hub cache directory: /home/jovyan/.cache/huggingface/hub
Checking cache directory for required files...
Cache check fa

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:08<00:00,  4.40s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:11<00:00,  5.69s/it]


Unsloth: Merge process complete. Saved to `/home/jovyan/model_local`
✅ 模型保存成功。

🔨 Step 2: 正在编译 llama.cpp 工具 (禁用 CURL 模式)...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 70969, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 70969 (delta 231), reused 163 (delta 162), pack-reused 70622 (from 2)
Receiving objects: 100% (70969/70969), 225.17 MiB | 41.36 MiB/s, done.
Resolving deltas: 100% (51182/51182), done.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMA

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  15%|█▌        |  306MB / 2.02GB,  153MB/s  
Processing Files (0 / 1):  15%|█▌        |  306MB / 2.02GB,  139MB/s  
Processing Files (0 / 1):  15%|█▌        |  309MB / 2.02GB,  129MB/s  
Processing Files (0 / 1):  15%|█▌        |  310MB / 2.02GB,  119MB/s  
Processing Files (0 / 1):  16%|█▌        |  315MB / 2.02GB,  113MB/s  
Processing Files (0 / 1):  16%|█▌        |  324MB / 2.02GB,  108MB/s  
Processing Files (0 / 1):  16%|█▋        |  330MB / 2.02GB,  103MB/s  
Processing Files (0 / 1):  17%|█▋        |  338MB / 2.02GB, 99.3MB/s  
Processing Files (0 / 1):  17%|█▋        |  350MB / 2.02GB, 97.3MB/s  
Processing Files (0 / 1):  18%|█▊        |  360MB / 2.02GB, 94.8MB/s  
Processing Files (0 / 1):  19%|█▊        |  376MB / 2.02GB, 94.0MB/s  
Processing Files (0 / 1):  19%|█▉        |  392MB / 2.02GB, 93.4MB/s  
Processing Files (0 / 1):  20%|██        |  412MB / 2.02GB, 93.6MB/s  
Processing


🎉🎉🎉 全部完成！
👉 你的模型地址: https://huggingface.co/Marcus719/Llama-3.2-3B-changedata-Lab2-GGUF
现在可以去创建 Hugging Face Space (Gradio) 来运行这个模型了！


In [24]:
if False:
 # I highly do NOT suggest - use Unsloth if possible
 from peft import AutoPeftModelForCausalLM
 from transformers import AutoTokenizer
 model = AutoPeftModelForCausalLM.from_pretrained(
 "lora_model", # YOUR MODEL YOU USED FOR TRAINING
 load_in_4bit = load_in_4bit,
 )
 tokenizer = AutoTokenizer.from_pretrained("lora_model")

In [25]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("Marcus719/Llama-3.2-Lab2", tokenizer, token="YOUR_HF_TOKEN_HERE"YOUR_HF_TOKEN_HERE"")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("Marcus719/Llama-3.2-Lab2", tokenizer, quantization_method = "f16", token="YOUR_HF_TOKEN_HERE"YOUR_HF_TOKEN_HERE"")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("Llama-3.2-3B-changedata-Lab2-GGUF", tokenizer, quantization_method = "q4_k_m", token="YOUR_HF_TOKEN_HERE")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
 model.push_to_hub_gguf(
 "hf/model", # Change hf to your username!
 tokenizer,
 quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
 token = "", # Get a token at https://huggingface.co/settings/tokens
 )